# CNN inicial: meu ponto de comparação

Vou construir uma primeira CNN para CIFAR-10 e registrar erros e acertos antes de avaliar mudanças na arquitetura.

**Ponto de partida:** [Classificador_Imagens-v1.ipynb](https://github.com/biaag10/machine_learning_projects/tree/develop/deep_learning/notebooks/Classificador_Imagens-v1.ipynb), do projeto de referência disponibilizado para estudo. Esta versão reorganiza a atividade, explicita decisões e deixa a interpretação dos resultados para a execução neste projeto.

**Como usar:** kernel da `.venv` de `deep_learning`; executar de cima para baixo. As saídas antigas foram removidas. Os resultados abaixo precisam ser gerados nesta execução.


In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
keras.utils.set_random_seed(SEED)
plt.rcParams.update({
    "figure.figsize": (9, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.18,
    "axes.prop_cycle": plt.cycler(color=["#0f766e", "#d97706", "#6366f1", "#be185d"]),
})
current = Path.cwd().resolve()
ROOT = next((p for p in [current, *current.parents]
             if (p / "module_deep_learning").is_dir()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra este notebook em deep_learning/notebooks e use a .venv do projeto.")
sys.path.insert(0, str(ROOT))
print("TensorFlow:", tf.__version__)
print("Python:", sys.executable)


def batches(X, y=None, batch_size=256, training=False):
    values = np.asarray(X, dtype="float32")
    tensors = values if y is None else (values, np.asarray(y))
    ds = tf.data.Dataset.from_tensor_slices(tensors)
    if training:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    options.threading.max_intra_op_parallelism = 1
    return ds.batch(batch_size).with_options(options).prefetch(1)


TensorFlow: 2.21.0
Python: c:\Users\Rafael Athaliba\Projetos Git\Machine_Learning\Machine_Learning_entrega\deep_learning\.venv\Scripts\python.exe


In [2]:
EPOCHS = 10
BATCH_SIZE = 64
VAL_SIZE = 0.20  # mesma divisão para as duas versões
MODEL_DIR = ROOT / "models" / "notebooks"
REPORT_DIR = ROOT / "reports" / "notebooks"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "cifar10_v1.keras"
RUN_NAME = "cifar10_v1"


## 1. Conhecendo a base e reservando a validação

CIFAR-10 contém imagens RGB de 32 × 32 pixels em dez classes. O teste oficial permanece separado. Reservo 20% do treino oficial para validação, com estratificação e semente 42. As duas versões repetem essa mesma divisão, antes de normalizar os pixels para [0,1]. A primeira execução pode baixar a base.


In [3]:
from sklearn.model_selection import train_test_split
import hashlib

(raw_train, labels_train), (raw_test, y_test) = keras.datasets.cifar10.load_data()
labels_train, y_test = labels_train.ravel(), y_test.ravel()
class_names = ["avião", "automóvel", "pássaro", "gato", "cervo",
               "cachorro", "sapo", "cavalo", "navio", "caminhão"]
train_idx, val_idx = train_test_split(np.arange(len(raw_train)), test_size=VAL_SIZE,
                                     stratify=labels_train, random_state=SEED)
X_train = raw_train[train_idx].astype("float32") / 255.0
X_val = raw_train[val_idx].astype("float32") / 255.0
X_test = raw_test.astype("float32") / 255.0
y_train, y_val = labels_train[train_idx], labels_train[val_idx]
split_signature = hashlib.sha256(np.asarray(train_idx, dtype="int64").tobytes()
                                  + np.asarray(val_idx, dtype="int64").tobytes()).hexdigest()
display(pd.DataFrame({"conjunto": ["treino", "validação", "teste"],
                      "imagens": [len(X_train),len(X_val),len(X_test)]}))
print("Assinatura da divisão:", split_signature)
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
examples = np.random.default_rng(SEED).choice(len(X_train), min(9,len(X_train)), replace=False)
for ax, i in zip(axes.ravel(), examples):
    ax.imshow(X_train[i]); ax.set_title(class_names[y_train[i]]); ax.axis("off")
fig.suptitle("Amostra do treino, escolhida com semente fixa"); fig.tight_layout(); plt.show()


 34906112/170498071 ━━━━━━━━━━━━━━━━━━━━ 18:24 8us/step

KeyboardInterrupt: 

## 2. Uma arquitetura de referência para começar

Mantenho os três blocos convolucionais de 16, 32 e 64 filtros da referência, seguidos de Flatten, Dense(64) e saída softmax. A convolução procura padrões locais; o pooling reduz a resolução. A saída escolhe entre dez classes, por isso uso sparse categorical cross-entropy.


In [ ]:
keras.utils.set_random_seed(SEED)
model = keras.Sequential([
    layers.Input(shape=(32,32,3)),
    layers.Conv2D(16, (3,3), activation="relu"), layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation="relu"), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation="relu"), layers.MaxPooling2D((2,2)),
    layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax"),
], name="cnn_inicial")
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 3. Treinamento e escolha do checkpoint

O checkpoint é escolhido pela acurácia de **validação**, nunca pela do teste. O notebook executa treinamento e pode demorar em CPU, especialmente na v2. Cada versão salva seu próprio arquivo, sem sobrescrever a MLP da Olist ou o modelo fornecido como referência.


In [ ]:
callbacks = [keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy", mode="max", save_best_only=True)]
history = model.fit(batches(X_train,y_train,BATCH_SIZE,training=True),
                    validation_data=batches(X_val,y_val,BATCH_SIZE),
                    epochs=EPOCHS, callbacks=callbacks, verbose=2, shuffle=False)
model = keras.models.load_model(MODEL_PATH)
training_history = {k: [float(v) for v in values] for k,values in history.history.items()}
(REPORT_DIR / f"{RUN_NAME}_history.json").write_text(json.dumps(training_history, indent=2), encoding="utf-8")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Acurácia", "Loss"]):
    ax.plot(np.arange(1,len(training_history[metric])+1),training_history[metric],label="Treino")
    ax.plot(np.arange(1,len(training_history[metric])+1),training_history[f"val_{metric}"],label="Validação")
    ax.set(title=title, xlabel="Época", ylabel=title); ax.legend()
fig.tight_layout(); plt.show()


## 4. Avaliação do checkpoint no teste

Só agora consulto o teste para relatar o resultado da configuração definida. Não vou trocar parâmetros nem escolher a melhor versão pelo teste e depois descrevê-lo como uma avaliação inédita. A softmax gera escores por classe; a maior saída define a previsão, sem garantir calibração.


In [ ]:
test_loss, test_accuracy = model.evaluate(batches(X_test,y_test,BATCH_SIZE), verbose=0)
probabilities = model.predict(batches(X_test,batch_size=BATCH_SIZE), verbose=0)
y_pred = probabilities.argmax(axis=1)
metrics = {"version": RUN_NAME, "test_loss": float(test_loss), "test_accuracy": float(test_accuracy),
           "seed": SEED, "validation_fraction": VAL_SIZE, "split_signature": split_signature,
           "epochs_requested": EPOCHS, "epochs_run": len(training_history["loss"]),
           "best_epoch": int(np.argmax(training_history["val_accuracy"])+1),
           "model_source": "treinado neste notebook; nao usa o arquivo anexado",
           "tensorflow_version": tf.__version__}
(REPORT_DIR / f"{RUN_NAME}_metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
print(f"Acurácia do checkpoint: {test_accuracy:.2%} | Loss: {test_loss:.4f}")
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
examples = np.random.default_rng(SEED).choice(len(X_test),min(9,len(X_test)),replace=False)
for ax, i in zip(axes.ravel(), examples):
    ax.imshow(X_test[i]); ax.axis("off")
    ax.set_title(f"Real: {class_names[y_test[i]]}\nPrevisto: {class_names[y_pred[i]]}",
                 color="#0f766e" if y_test[i]==y_pred[i] else "#be185d", fontsize=9)
fig.suptitle("Amostra fixa do teste: acertos e erros"); fig.tight_layout(); plt.show()


## 5. Onde a rede se confunde?

Cada linha da matriz corresponde à classe real. A diagonal representa os acertos; as demais células mostram confusões. Vou olhar a distribuição dos erros antes de concluir que a acurácia geral resume bem o resultado.


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred, labels=np.arange(10))
denominator = cm.sum(axis=1, keepdims=True)
normalized = np.divide(cm, denominator, out=np.zeros_like(cm,dtype=float), where=denominator!=0)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(normalized,cmap="Blues",vmin=0,vmax=1)
fig.colorbar(im,ax=ax,label="Proporção na classe real")
ax.set_xticks(np.arange(10),class_names,rotation=45,ha="right")
ax.set_yticks(np.arange(10),class_names)
ax.set(xlabel="Classe prevista",ylabel="Classe real",title=f"Matriz de confusão — {RUN_NAME}")
for i in range(10):
    for j in range(10):
        ax.text(j,i,f"{cm[i,j]}\n{normalized[i,j]:.0%}",ha="center",va="center",fontsize=7,
                color="white" if normalized[i,j]>.5 else "black")
fig.tight_layout(); plt.show()
per_class = pd.DataFrame({"classe": class_names,"total_teste":cm.sum(axis=1),
                          "fracao_acertada":np.diag(normalized)})
display(per_class)
per_class.to_csv(REPORT_DIR / f"{RUN_NAME}_por_classe.csv",index=False)


## Minhas conclusões após executar

- Que diferença observei entre as curvas de treino e validação?
- Quais classes foram mais difíceis para esta rede?
- Que pares de classes concentraram confusões?
- O que os exemplos visuais ajudam a perceber que a acurácia não mostra?
- Que conclusão ainda precisaria de outro experimento ou outra semente?

Os resultados da referência e do modelo anexado não foram transportados para este notebook. Vou registrar os meus resultados após executar a versão identificada acima.
